# Fixed-charge/interface-SRH coupling — release 0.4

This notebook replaces the prescribed front-surface recombination velocity by
the chain

$$Q_f\rightarrow\psi_s\rightarrow(n_s,p_s)\rightarrow
U_s(\Delta n_s)\rightarrow\Delta n(x).$$

The surface electrostatics uses Fermi–Dirac statistics and can be recalculated
self-consistently at each injection level. The interface kinetics uses a
distributed SRH model and is coupled as a nonlinear boundary condition to the
steady-state 1D transport solver.

Important limitations: $Q_{it}=0$, there is no Fermi-level pinning, the
Adamowicz parameters are not measurements of POx/InP, and the reported
``PL proxy`` is internal radiative recombination rather than detected PL.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

root = Path.cwd()
if not (root / "src").exists(): root = root.parent
sys.path.insert(0, str(root / "src"))
from surpass import (ADAMOWICZ_INP, BulkModel, BulkTransportTable,
                        InterfaceDefectModel, Layer, OpticalStack,
                        SteadyState1DSolver, SurfaceBoundaryTable,
                        SurfaceSRHModel, Wafer, get_optical_material)

try:
    import scienceplots
    plt.style.use(["science", "notebook", "no-latex"])
except ImportError:
    plt.style.use("default")
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

## 1. Editable physical assumptions

In [ ]:
wavelength_nm = 514.0
thickness_um = 620.0
thickness_cm = thickness_um*1e-4
tau_bulk_srh_s = 100e-9
rear_s_cm_s = 1e5
enable_bgn = False
qf_values = np.linspace(-6e12, 6e12, 13)  # stack charge + applied corona charge

defects = InterfaceDefectModel(
    dit_mid_ev1_cm2=1e11,
    sigma_n_cm2=1e-14,
    sigma_p_cm2=1e-13,
    vth_n_cm_s=4.13e7,
    vth_p_cm_s=1.51e7,
    shape="constant",
    energy_points=301,
)
print(defects.source)

wafer_settings = {
    "n-InP": dict(wafer=Wafer("InP", donor_cm3=8.5e18, thickness_um=thickness_um),
                  intensity=1.0, color="tab:blue"),
    "p-InP": dict(wafer=Wafer("InP", acceptor_cm3=5.1e18, thickness_um=thickness_um),
                  intensity=13.0, color="tab:red"),
}

air, inp = get_optical_material("air"), get_optical_material("InP_demo")
pox, alox = get_optical_material("POx_demo"), get_optical_material("AlOx_demo")
stack = OpticalStack(air, [Layer(pox, 10.0, "POx"), Layer(alox, 10.0, "AlOx")], inp)
edges_cm = np.r_[0.0, np.geomspace(1e-8, 5e-4, 140),
                 np.linspace(5e-4, thickness_cm, 161)[1:]]
centres_um = 0.5*(edges_cm[:-1]+edges_cm[1:])*1e4

## 2. Surface-only charge response

The reference injection below is used only to visualize the boundary model.
The coupled solver later determines the actual surface injection itself.

In [ ]:
delta_reference = 1e11
surface_rows = []
surface_models = {}
for label, settings in wafer_settings.items():
    bulk = BulkModel(settings["wafer"], enable_bgn=enable_bgn, tau_srh_s=tau_bulk_srh_s)
    for qf in qf_values:
        model = SurfaceSRHModel(bulk, qf, defects,
                                electrostatics="self_consistent",
                                driving_force="excess_product")
        state = model.state(delta_reference)
        surface_models[(label, qf)] = model
        surface_rows.append({"wafer": label, "Qf (cm^-2)": qf,
                             "psi_s (V)": state.psi_surface_v,
                             "n_s (cm^-3)": state.n_surface_cm3,
                             "p_s (cm^-3)": state.p_surface_cm3,
                             "U_s (cm^-2 s^-1)": state.recombination_cm2_s,
                             "S_eff=U_s/dn (cm/s)": state.effective_s_cm_s})
surface_df = pd.DataFrame(surface_rows)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.7))
for label, settings in wafer_settings.items():
    subset = surface_df[surface_df.wafer == label]
    axes[0].plot(subset["Qf (cm^-2)"]/1e12, subset["psi_s (V)"],
                 marker="o", label=label, color=settings["color"])
    axes[1].semilogy(subset["Qf (cm^-2)"]/1e12,
                     subset["S_eff=U_s/dn (cm/s)"], marker="o",
                     label=label, color=settings["color"])
axes[0].set(xlabel="$Q_f/q$ ($10^{12}$ cm$^{-2}$)", ylabel="$\\psi_s$ (V)")
axes[1].set(xlabel="$Q_f/q$ ($10^{12}$ cm$^{-2}$)", ylabel="$U_s/\\Delta n_b$ (cm s$^{-1}$)")
axes[0].legend(); axes[1].legend(); fig.tight_layout()

The effective velocity above is normalized to the **bulk-side** excess density.
It may greatly exceed a microscopic capture velocity in strong inversion,
because field effect concentrates the minority carrier at the surface. In the
transport solution, this does not imply an unlimited rate: the surface excess
density is depleted until diffusive supply balances $U_s$.

## 3. Couple every fixed-charge point to optical generation and transport

In [ ]:
bulk_solvers, generations = {}, {}
for label, settings in wafer_settings.items():
    bulk = BulkModel(settings["wafer"], enable_bgn=enable_bgn, tau_srh_s=tau_bulk_srh_s)
    bulk_solvers[label] = SteadyState1DSolver(BulkTransportTable.from_bulk_model(
        bulk, delta_min_cm3=1.0, delta_max_cm3=1e21, points=261))
    generations[label] = stack.generation_cell_average_cm3_s(
        wavelength_nm, settings["intensity"], edges_cm)

coupled_results, coupled_rows = {}, []
for label, settings in wafer_settings.items():
    for qf in qf_values:
        boundary = SurfaceBoundaryTable.from_surface_model(
            surface_models[(label, qf)], delta_min_cm3=1.0,
            delta_max_cm3=1e21, points=181, label=f"{label}, Qf={qf:.2e}")
        result = bulk_solvers[label].solve(
            thickness_cm, generations[label], boundary, rear_s_cm_s,
            cell_edges_cm=edges_cm, max_nfev=1500)
        coupled_results[(label, qf)] = result
        coupled_rows.append({
            "wafer": label, "Qf (cm^-2)": qf,
            "<dn> (cm^-3)": result.average_delta_n_cm3,
            "dn_surface (cm^-3)": result.front_surface_delta_cm3,
            "effective Sf (cm/s)": result.front_effective_s_cm_s,
            "tau_eff (ns)": result.effective_lifetime_s*1e9,
            "surface fraction": result.front_surface_recombination_cm2_s/result.generated_flux_cm2_s,
            "bulk radiative fraction": result.internal_radiative_yield,
            "internal radiative flux": result.bulk_radiative_recombination_cm2_s,
            "balance error": result.relative_balance_error,
            "converged": result.converged,
        })
coupled_df = pd.DataFrame(coupled_rows)
if not coupled_df.converged.all():
    print("WARNING: unconverged charge points:")
    display(coupled_df.loc[~coupled_df.converged,
                           ["wafer", "Qf (cm^-2)", "balance error"]])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True)
for label, settings in wafer_settings.items():
    subset = coupled_df[coupled_df.wafer == label].copy()
    q = subset["Qf (cm^-2)"]/1e12
    reference = float(subset.loc[np.isclose(subset["Qf (cm^-2)"], 0),
                                 "internal radiative flux"].iloc[0])
    axes[0, 0].plot(q, subset["internal radiative flux"]/reference,
                    marker="o", label=label, color=settings["color"])
    axes[0, 1].semilogy(q, subset["effective Sf (cm/s)"],
                        marker="o", label=label, color=settings["color"])
    axes[1, 0].plot(q, subset["tau_eff (ns)"],
                    marker="o", label=label, color=settings["color"])
    axes[1, 1].plot(q, subset["surface fraction"],
                    marker="o", label=label, color=settings["color"])
axes[0, 0].set(ylabel="Internal radiative flux / value at $Q_f=0$")
axes[0, 1].set(ylabel="Coupled effective $S_f$ (cm s$^{-1}$)")
axes[1, 0].set(xlabel="$Q_f/q$ ($10^{12}$ cm$^{-2}$)", ylabel="Effective lifetime (ns)")
axes[1, 1].set(xlabel="$Q_f/q$ ($10^{12}$ cm$^{-2}$)", ylabel="Front-surface recombination fraction")
axes[0, 0].legend(); axes[0, 1].legend(); fig.tight_layout()

## 4. Carrier profiles at negative, zero, and positive charge

In [ ]:
selected_qf = [-4e12, 0.0, 4e12]
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for row, (label, settings) in enumerate(wafer_settings.items()):
    for qf in selected_qf:
        result = coupled_results[(label, qf)]
        axes[row, 0].semilogx(result.depth_cm*1e4, result.delta_n_cm3,
                              label=f"$Q_f/q$={qf/1e12:+g}")
        axes[row, 1].plot(result.depth_cm*1e4, result.delta_n_cm3,
                         label=f"$Q_f/q$={qf/1e12:+g}")
    axes[row, 0].set(title=f"{label}: near front", xlim=(1e-4, 20),
                     xlabel="Depth (µm)", ylabel="$\\Delta n$ (cm$^{-3}$)")
    axes[row, 1].set(title=f"{label}: full wafer", xlim=(0, thickness_um),
                     xlabel="Depth (µm)", ylabel="$\\Delta n$ (cm$^{-3}$)")
axes[0, 0].legend(fontsize=8); fig.tight_layout()

## 5. Is illumination-dependent band bending important here?

In [ ]:
injection_sweep = np.logspace(6, 17, 80)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.7))
comparison_rows = []
for label, settings in wafer_settings.items():
    bulk = BulkModel(settings["wafer"], enable_bgn=enable_bgn, tau_srh_s=tau_bulk_srh_s)
    for qf in (-4.2e12, 4.2e12):
        equilibrium = SurfaceSRHModel(bulk, qf, defects, electrostatics="equilibrium")
        self_consistent = SurfaceSRHModel(bulk, qf, defects, electrostatics="self_consistent")
        eq_states = [equilibrium.state(d) for d in injection_sweep]
        sc_states = [self_consistent.state(d) for d in injection_sweep]
        key = f"{label}, {qf/1e12:+.1f}e12"
        axes[0].semilogx(injection_sweep,
                         [s.psi_surface_v for s in sc_states], label=key)
        axes[0].semilogx(injection_sweep,
                         [s.psi_surface_v for s in eq_states], ls="--", color=axes[0].lines[-1].get_color())
        axes[1].loglog(injection_sweep,
                       [s.recombination_cm2_s for s in sc_states], label=key)
        axes[1].loglog(injection_sweep,
                       [s.recombination_cm2_s for s in eq_states], ls="--", color=axes[1].lines[-1].get_color())
axes[0].set(xlabel="$\\Delta n_b$ (cm$^{-3}$)", ylabel="$\\psi_s$ (V)")
axes[1].set(xlabel="$\\Delta n_b$ (cm$^{-3}$)", ylabel="$U_s$ (cm$^{-2}$ s$^{-1}$)")
axes[0].legend(fontsize=7); axes[1].legend(fontsize=7); fig.tight_layout()

Solid curves recalculate the space charge at every injection; dashed curves
freeze the equilibrium band bending. Their agreement or divergence gives a
direct test of whether the low-injection electrostatic approximation is valid.

## Interpretation and next validation priorities

- The model now predicts the injection level instead of imposing it, and fixed
  charge enters through the physical surface boundary rather than through a
  post-processed $J_{0s}$.
- It should reproduce the qualitative corona trend: negative added charge
  increases the n-InP surface loss and decreases the p-InP loss over the
  relevant branch, while turning points remain possible.
- Absolute PL changes remain uncertain because $D_{it}(E)$, capture cross
  sections, the bulk coefficients, and photon recycling are not independently
  known.
- Interface-state charge is currently omitted. Adding amphoteric $Q_{it}$ and
  solving $Q_f+Q_{it}+Q_{sc}=0$ is the most important electrostatic extension
  if pinning is appreciable.
- The next numerical layer should be time dependent. Before fitting biexponential
  TRPL, however, bulk coefficients and the optical escape/recycling model should
  be constrained, otherwise multiple parameter combinations will be
  indistinguishable.